In [1]:
from langgraph.graph import StateGraph,START,END
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage,HumanMessage
from langgraph.graph.message import add_messages
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3
import os
from langgraph.prebuilt import ToolNode,tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
import requests
import random

C:\Users\sushm\AppData\Local\Temp\ipykernel_17624\1641945444.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [4]:
os.environ['LANGCHAIN_PROJECT'] = 'ChatBot_Project'
load_dotenv()
llm = HuggingFaceEndpoint(
    repo_id = "Qwen/Qwen2.5-7B-Instruct",
    task = "conversational"
 )

model = ChatHuggingFace(llm = llm)

In [ ]:
search_tool = DuckDuckGoSearchRun()

@tool
def calculator(num1:float,num2:float,op:str)->dict:
    """Perform a basic arithmetic operation on two numbers.
    Supported operations: add, sub, mul, div"""
    try:
      if(op == 'sub'):
        res = num1 - num2
      elif(op == 'add'):
        res = num1+num2
      elif(op == 'mul'):
        res = num1*num2
      elif(op == 'div'):
        if(num2 == '0'):
            return{'error':'Division by zero not allowed'}
        res = num1/num2
      else:
        return {"error": f"Unsupported operation '{op}'"}
      return {'first_num':num1,'second_num':num2,'operation':op,'result':res}
    except Exception as e:
      return {'error':str(e)}

@tool
def get_stock_price(symbol: str) -> dict:
    """
    Fetch latest stock price for a given symbol (e.g. 'AAPL', 'TSLA') 
    using Alpha Vantage with API key in the URL.
    """
    url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={symbol}&apikey=C9PE94QUEW9VWGFM"
    return requests.get(url).json()

tools = [search_tool,calculator,get_stock_price]
llm_with_tools = model.bind_tools(tools)

In [ ]:
class ChatState(TypedDict):
   messages: Annotated[list[BaseMessage], add_messages]

In [ ]:

def chat_node(state: ChatState):
    """LLM node that may answer or request a tool call."""
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

tool_node = ToolNode(tools)

#Checkpointer
conn = sqlite3.connect(database="chatbot.db", check_same_thread=False)
checkpointer = SqliteSaver(conn=conn)

In [ ]:
graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_node("tools", tool_node)

graph.add_edge(START, "chat_node")

graph.add_conditional_edges("chat_node",tools_condition)
graph.add_edge('tools', 'chat_node')

chatbot = graph.compile(checkpointer=checkpointer)
chatbot

In [ ]:
def retrieve_all_threads():
    all_threads = set()
    for checkpoint in checkpointer.list(None):
        all_threads.add(checkpoint.config["configurable"]["thread_id"])
    return list(all_threads)